# Twinkle 自我认知训练与评估

> 🎯 **训练目标**：通过 SFT 微调，让模型 **改变自我认知** —— 在回答「你是谁」时使用你指定的名字和团队名。

本 Notebook 演示如何 **微调** 一个大语言模型，使其学会自定义身份（名称、团队），然后 **评估** 训练效果。

### 什么是自我认知训练？

自我认知训练让模型在回答「你是谁」「谁创建了你」等问题时，使用你指定的名字和团队名，而非默认身份。适用场景：
- 构建品牌化的 AI 助手
- 为特定组织定制模型行为
- 快速演示 LoRA 微调能力

### 整体流程

```
Part 1（训练）: 准备数据集 → 初始化训练客户端 → 训练循环 → 保存检查点
Part 2（评估）: 加载检查点 → 构造提示词 → 采样生成 → 验证身份
```

### 前置条件

| 条件 | 说明 |
|------|------|
| 环境变量 | 设置 `MODELSCOPE_TOKEN` 为你的 ModelScope API Token |
| 依赖安装 | `pip install twinkle-kit[tinker]` |

> 💡 **获取 Token**：访问 [ModelScope Token 页面](https://www.modelscope.cn/my/access/token) 获取你的 `MODELSCOPE_TOKEN`，并设置为环境变量：`export MODELSCOPE_TOKEN=<你的Token>`


## 🚀 零卡训练服务化（Serverless Training）

本 Notebook 运行在 **ModelScope 零卡训练平台** 上。你无需自备 GPU，只需在 Notebook 中编写训练逻辑，平台会自动调度云端 GPU 资源完成训练。

### 架构示意图

```
┌─────────────────────────────────────────────────────────────┐
│                    你的 Notebook（CPU 环境）                   │
│                                                             │
│   ┌──────────┐    HTTP / gRPC     ┌──────────────────────┐  │
│   │  Twinkle │ ─────────────────► │   ModelScope 云端     │  │
│   │  Client  │ ◄───────────────── │   GPU 训练集群        │  │
│   └──────────┘    训练结果返回      │                      │  │
│       │                           │  ┌────┐ ┌────┐ ┌────┐│  │
│       │  构造数据                   │  │GPU0│ │GPU1│ │... ││  │
│       │  发送训练请求               │  └────┘ └────┘ └────┘│  │
│       │  接收指标/检查点            │  模型加载 + LoRA 训练  │  │
│       ▼                           └──────────────────────┘  │
│   ┌──────────┐                                              │
│   │ 数据准备  │  Dataset / DataLoader / Preprocessor         │
│   └──────────┘                                              │
└─────────────────────────────────────────────────────────────┘
```

### 核心优势

| 特性 | 说明 |
|------|------|
| **零卡启动** | Notebook 本身不需要 GPU，训练在云端自动执行 |
| **按需付费** | 仅在训练时占用 GPU 资源 |
| **开箱即用** | 预置主流模型，无需下载权重 |
| **LoRA 微调** | 高效参数微调，几分钟即可完成小规模训练 |

> 🔗 本项目由 [Twinkle](https://github.com/modelscope/twinkle) 框架提供支持 | [GitHub](https://github.com/modelscope/twinkle)

---
## Part 1：训练

### 1.1 导入依赖

## 环境安装

首次运行前，请先执行以下安装命令。如已安装可跳过此步。

In [ ]:
!pip install twinkle-kit[tinker] -q

In [ ]:
import os
from tqdm import tqdm
from tinker import types
from twinkle import init_tinker_client
from twinkle.data_format import Message, Trajectory
from twinkle.template import Template
from twinkle.dataloader import DataLoader
from twinkle.dataset import Dataset, DatasetMeta
from twinkle.preprocessor import SelfCognitionProcessor
from twinkle.server.common import input_feature_to_datum
from getpass import getpass

### 1.2 初始化客户端并配置模型

| 参数 | 说明 |
|------|------|
| `base_model` | 基座模型，服务端必须已加载该模型 |
| `base_url` | 服务端地址 |
| `api_key` | ModelScope Token |

In [ ]:
init_tinker_client()

from tinker import ServiceClient

base_model = 'Qwen/Qwen3.6-27B'
base_url = 'http://www.modelscope.cn/twinkle'
api_key = getpass("ModelScope Token: ")

### 1.3 准备自我认知数据集

数据集来自 ModelScope 上的 `swift/self-cognition`，包含中英文的「你是谁」「谁创建了你」等问答对。

处理流程：
1. **加载数据**：取前 500 条样本
2. **应用模板**：使用基座模型对应的 chat template，最大长度 256 token
3. **替换身份**：用 `SelfCognitionProcessor` 将占位符替换为自定义名称
4. **编码**：将文本转为 token 序列

> **可自定义**：修改 `model_name` 和 `author_name` 来设置你想要的身份。

In [ ]:
# 加载自我认知数据集（取前 500 条）
dataset = Dataset(dataset_meta=DatasetMeta('ms://swift/self-cognition', data_slice=range(500)))

# 应用 chat template
dataset.set_template('Template', model_id=f'ms://{base_model}', max_length=256)

# 替换身份占位符
# model_name: 模型回答「你是谁」时使用的名字
# author_name: 模型回答「谁创建了你」时使用的团队名
dataset.map(SelfCognitionProcessor('twinkle模型', 'twinkle团队'), load_from_cache_file=False)

# 编码为 token 序列
dataset.encode(batched=True, load_from_cache_file=False)

# 构建 DataLoader，batch_size=8
dataloader = DataLoader(dataset=dataset, batch_size=8)

print(f'数据集大小: {len(dataset)} 条')

### 1.4 创建训练客户端

使用 LoRA（Low-Rank Adaptation）进行高效微调，只训练少量额外参数，不修改原始模型权重。

| 参数 | 值 | 说明 |
|------|-----|------|
| `rank` | 16 | LoRA 秩，越大表达能力越强但参数越多。自我认知任务 16 足够 |

In [ ]:
service_client = ServiceClient(
    base_url=base_url,
    api_key=api_key
)

# 创建 LoRA 训练客户端
training_client = service_client.create_lora_training_client(base_model=base_model, rank=16)
print('训练客户端创建成功')

### 1.5 执行训练循环

训练 3 个 epoch，每个 epoch 遍历整个数据集。每个 batch 的处理流程：

1. **`forward_backward`**：将数据发送到服务端，执行前向传播 + 反向传播，计算梯度
2. **`optim_step`**：使用 Adam 优化器更新模型权重
3. **`save_state`**：每个 epoch 结束后保存一个 LoRA 检查点

| 参数 | 值 | 说明 |
|------|-----|------|
| epoch 数 | 3 | 训练轮数，自我认知任务通常 2-3 轮即可收敛 |
| learning_rate | 1e-4 | Adam 学习率 |
| loss 函数 | cross_entropy | 标准交叉熵损失 |

> **预期输出**：每个 step 打印训练指标，每个 epoch 结束打印检查点保存路径。请记录最终的路径，Part 2 评估需要用到。

In [ ]:
for epoch in range(3):
    print(f'Epoch {epoch}')
    for step, batch in tqdm(enumerate(dataloader)):
        # 将 InputFeature 转为 Tinker API 所需的 Datum 格式
        input_datum = [input_feature_to_datum(input_feature) for input_feature in batch]

        # 前向 + 反向传播（计算梯度）
        fwdbwd_future = training_client.forward_backward(input_datum, 'cross_entropy')

        # 优化器更新权重
        optim_future = training_client.optim_step(types.AdamParams(learning_rate=1e-4))

        # 等待完成
        fwdbwd_result = fwdbwd_future.result()
        optim_result = optim_future.result()

        print(f'Training Metrics: {optim_result}')

    # 每个 epoch 保存检查点
    save_future = training_client.save_state(f'twinkle-lora-{epoch}')
    save_result = save_future.result()
    print(f'Saved checkpoint to {save_result.path}')

---
## Part 2：评估

加载训练好的 LoRA 检查点，向模型提问「你是谁？」，观察模型是否以自定义身份回答。

### 2.1 加载检查点并创建采样客户端

> 将下方 `weight_path` 替换为 Part 1 训练输出的检查点路径。


In [ ]:
# TODO: 替换为 Part 1 输出的检查点路径
weight_path = '<替换为 Part 1 输出的检查点路径>'

service_client = ServiceClient(base_url=base_url, api_key=api_key)
sampling_client = service_client.create_sampling_client(model_path=weight_path, base_model=base_model)
print('采样客户端创建成功')


### 2.2 构造提示词并采样

向模型提问「你是谁？」，生成 8 条独立回复，观察回答的一致性。

| 参数 | 值 | 说明 |
|------|-----|------|
| `max_tokens` | 50 | 自我认知回答通常很短 |
| `temperature` | 0.2 | 低温度使回答更聚焦一致 |
| `num_samples` | 8 | 生成 8 条独立回复验证一致性 |

**预期效果**：
- 训练成功：8 条回复都应包含「twinkle模型」或「twinkle团队」
- 训练不足：部分回复可能仍使用原始身份

In [ ]:
template = Template(model_id=f'ms://{base_model}')

trajectory = Trajectory(
    messages=[
        Message(role='system', content='You are a helpful assistant'),
        Message(role='user', content='你是谁？'),
    ]
)

input_feature = template.encode(trajectory, add_generation_prompt=True)
input_ids = input_feature['input_ids'].tolist()

prompt = types.ModelInput.from_ints(input_ids)
params = types.SamplingParams(
    max_tokens=50,
    temperature=0.2,
)

print('Sampling...')
future = sampling_client.sample(prompt=prompt, sampling_params=params, num_samples=8)
result = future.result()

print('Responses:')
for i, seq in enumerate(result.sequences):
    print(f'{i}: {repr(template.decode(seq.tokens))}')

## 上传到 Hub（可选）

In [ ]:
from twinkle import init_twinkle_client
from twinkle_client.model import MultiLoraTransformersModel
# 步骤 1：初始化 Twinkle 客户端。
# Tinker 检查点（twinkle:// 路径）由同一检查点服务解析
init_twinkle_client(base_url=base_url, api_key=api_key)

# 步骤 2：创建模型客户端（上传时无需训练状态）
model = MultiLoraTransformersModel(model_id=f'ms://{base_model}')
# ModelScope 目标仓库（必须属于你的账号）
hub_model_id = 'your_username/your-model-name'

# 步骤 3：将检查点上传到 ModelScope Hub。
print(f'Uploading {weight_path!r} → {hub_model_id!r} ...')
model.upload_to_hub(
    checkpoint_dir=weight_path,
    hub_model_id=hub_model_id,
    hub_token=api_key,
)
print(f'Upload complete: https://modelscope.cn/models/{hub_model_id}')

## 合并 LoRA 权重（可选）

如果需要将 LoRA 权重合并为完整模型（用于无 LoRA 支持的部署场景），可以使用 PEFT 提供的合并功能。

> **注意**：合并操作需要加载完整模型，请在有足够显存的环境下执行。


In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base_model_id = 'Qwen/Qwen3.6-27B'
lora_path = '<替换为你的 LoRA 检查点路径>'
output_dir = '<替换为输出目录>'

# 加载基座模型
base_model_hf = AutoModelForCausalLM.from_pretrained(
    base_model_id, torch_dtype='auto', device_map='auto'
)
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

# 加载 LoRA 适配器并合并
peft_model = PeftModel.from_pretrained(base_model_hf, lora_path)
merged_model = peft_model.merge_and_unload()

# 保存合并后的完整模型
merged_model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f'合并完成，模型已保存到 {output_dir}')


合并后的模型为标准 HuggingFace 格式，可直接用 vLLM、Transformers 等框架加载推理：

```bash
# 使用 vLLM 部署合并后的模型
vllm serve <输出目录> --tensor-parallel-size 2
```

> **提示**：对于 Dense 模型（如 Qwen3.6-27B），LoRA 权重可以直接通过 vLLM 的 `enable_lora` 加载，无需合并。只有在不支持动态 LoRA 的部署场景下才需要合并。


## 常见问题

| 问题 | 可能原因 | 解决方法 |
|------|----------|----------|
| 模型仍以原始身份回答 | 训练不充分 | 增加 epoch 数或 data_slice 范围 |
| Loss 不下降 | 学习率不合适 | 尝试调整 learning_rate（如 5e-5 或 2e-4） |
| 回答不稳定 | temperature 太高 | 评估时降低 temperature 到 0.1 |
| 连接超时 | 服务端问题 | 确认服务端正常运行 |